In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
 99% 329M/331M [00:00<00:00, 721MB/s]
100% 331M/331M [00:00<00:00, 736MB/s]


In [3]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [4]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [6]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [7]:
len(age)

23708

In [8]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [9]:
df.shape

(23708, 3)

In [10]:
df.head()

,age,gender,img
0,53,1,53_1_0_20170104184642790.jpg.chip.jpg
1,45,0,45_0_3_20170119171404704.jpg.chip.jpg
2,44,0,44_0_0_20170113210319338.jpg.chip.jpg
3,40,0,40_0_4_20170104204553772.jpg.chip.jpg
4,31,1,31_1_4_20170112235504601.jpg.chip.jpg


In [11]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [12]:
train_df.shape

(20000, 3)

In [13]:
test_df.shape

(3708, 3)

In [14]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [15]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [17]:
resnet = ResNet50( weights='imagenet',include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [18]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [19]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [21]:
# Create a custom generator function to yield individual data points
def unbatched_generator(generator):
    for batch_x, batch_y in generator:
        for i in range(batch_x.shape[0]):
            yield batch_x[i], {'age': batch_y[0][i], 'gender': batch_y[1][i]}

# Create tf.data.Dataset from the unbatched custom generators and then batch
train_ds = tf.data.Dataset.from_generator(
    lambda: unbatched_generator(train_generator),
    output_signature=(
        tf.TensorSpec(shape=(200, 200, 3), dtype=tf.float32),
        {'age': tf.TensorSpec(shape=(), dtype=tf.int64), # Shape is empty for a single scalar
         'gender': tf.TensorSpec(shape=(), dtype=tf.int64)} # Shape is empty for a single scalar
    )
).batch(train_generator.batch_size).prefetch(tf.data.AUTOTUNE)


test_ds = tf.data.Dataset.from_generator(
    lambda: unbatched_generator(test_generator),
    output_signature=(
        tf.TensorSpec(shape=(200, 200, 3), dtype=tf.float32),
        {'age': tf.TensorSpec(shape=(), dtype=tf.int64), # Shape is empty for a single scalar
         'gender': tf.TensorSpec(shape=(), dtype=tf.int64)} # Shape is empty for a single scalar
    )
).batch(test_generator.batch_size).prefetch(tf.data.AUTOTUNE)

# Calculate steps per epoch
steps_per_epoch = train_generator.samples // train_generator.batch_size
validation_steps = test_generator.samples // test_generator.batch_size

model.fit(train_ds, epochs=20, validation_data=test_ds, steps_per_epoch=steps_per_epoch, validation_steps=validation_steps)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 249s 373ms/step - age_loss: 16.3359 - age_mae: 16.3359 - gender_accuracy: 0.5098 - gender_loss: 1.5478 - loss: 169.5705 - val_age_loss: 14.5443 - val_age_mae: 14.5443 - val_gender_accuracy: 0.5264 - val_gender_loss: 0.6919 - val_loss: 83.0466
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 260s 416ms/step - age_loss: 14.9776 - age_mae: 14.9776 - gender_accuracy: 0.5255 - gender_loss: 0.6921 - loss: 83.4930 - val_age_loss: 15.4237 - val_age_mae: 15.4237 - val_gender_accuracy: 0.5250 - val_gender_loss: 0.6921 - val_loss: 83.9394
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 262s 420ms/step - age_loss: 14.8911 - age_mae: 14.8911 - gender_accuracy: 0.5210 - gender_loss: 0.6939 - loss: 83.5840 - val_age_loss: 14.3088 - val_age_mae: 14.3088 - val_gender_accuracy: 0.5228 - val_gender_loss: 0.6924 - val_loss: 82.8543
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 262s 420ms/step - age_loss: 15.0627 - age_mae: 15.0627 - gender_accuracy: 0.5268 - gender_loss: 0.6921 - loss: 83